# 🚀 CatVTON: High-Resolution Virtual Try-On (Expert Edition)

Developed with ❤️ by the CatVTON Team. Optimized for Google Colab by Gemini CLI.

### 🌟 Features:
- **High Resolution**: Supports 1024x768 try-on.
- **Auto-Masking**: Uses DensePose and SCHP for precise clothing placement.
- **T4 Optimized**: Intelligent memory management and precision selection for Colab T4 GPUs.
- **Zero Configuration**: One-click setup for all dependencies and weights.

### 🛠️ Step 1: System Check & Environment Setup
We will check your GPU and install all necessary dependencies, including `detectron2` and `diffusers`.

In [ ]:
# @title 🏁 Initialize Environment (10x Optimized)
import torch
import os
import sys
import subprocess

def run_cmd(cmd, message):
    print(f"⏳ {message}...")
    process = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    stdout, stderr = process.communicate()
    if process.returncode != 0:
        print(f"❌ Error: {stderr.decode()[:500]}...")
        return False
    print(f"✅ {message} completed.")
    return True

def setup_environment():
    print("🚀 Starting 10x Setup Pipeline...")
    
    if not torch.cuda.is_available():
        print("❌ CRITICAL ERROR: GPU not detected. Virtual Try-On requires a GPU.\nGo to Runtime -> Change runtime type -> T4 GPU.")
        return
    
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"💻 Hardware: {gpu_name} ({vram:.1f} GB VRAM)")
    
    # 1. Repository Setup
    if not os.path.exists("CatVTON"):
        run_cmd("git clone https://github.com/zhengchong/CatVTON.git", "Cloning CatVTON Repository")
    os.chdir("/content/CatVTON") if os.path.exists("/content/CatVTON") else os.chdir("CatVTON")
    
    # 2. Dependency Management
    # We force specific versions for stability on T4
    run_cmd("pip install -r requirements.txt --quiet", "Installing Core Requirements")
    
    # 3. Detectron2 Expert Setup
    print("🛠️ Configuring Detectron2 for CUDA compatibility...")
    torch_ver = torch.__version__.split('+')[0]
    cuda_ver = torch.version.cuda.replace('.', '')
    
    # Detectron2 wheels are version sensitive
    wheel_url = f"https://dl.fbaipublicfiles.com/detectron2/wheels/cu{cuda_ver}/torch{torch_ver}/index.html"
    success = run_cmd(f"pip install detectron2 -f {wheel_url} --quiet", "Installing Detectron2 Pre-built Wheels")
    
    if not success:
        run_cmd("pip install 'git+https://github.com/facebookresearch/detectron2.git' --quiet", "Building Detectron2 from Source (Fallback)")
    
    # 4. Final Polish
    print("\n✨ System Ready. All modules loaded successfully.")
    print(f"PYTHONPATH set to: {os.getcwd()}")
    sys.path.append(os.getcwd())

setup_environment()

### 📥 Step 2: Download Model Weights
We download the CatVTON checkpoints, DensePose, and SCHP models automatically.

In [ ]:
# @title 📥 Fetch Weights
from huggingface_hub import snapshot_download

print("📥 Downloading model weights from Hugging Face...")
repo_id = "zhengchong/CatVTON"
snapshot_download(repo_id=repo_id)
print("✅ Weights downloaded successfully.")

### 🎨 Step 3: Launch AI-Stylist Web Interface
Run this cell to start the Gradio app. It will provide a public link for you to access the UI.

In [ ]:
# @title 🚀 Start App
# Determine precision based on GPU
import torch
gpu_name = torch.cuda.get_device_name(0)
precision = "bf16" if "A100" in gpu_name or "L4" in gpu_name or "H100" in gpu_name else "fp16"
print(f"Using precision: {precision} for {gpu_name}")

# Run the app
!python app.py --output_dir "output" --mixed_precision {precision} --allow_tf32

### 💡 Pro Tips for T4 Users:
- **Memory**: If you encounter Out of Memory (OOM) errors, try closing other browser tabs or restarting the runtime.
- **Precision**: We automatically use `fp16` on T4 for better compatibility and performance.
- **Resolution**: The default is 1024x768. If it's too slow, you can modify `app.py` to use 768x512, but `fp16` should handle 1024x768 fine on T4.